## Fine-tuning BERT do klasyfikacji triage

### 📋 Cel notebooka
Ten notebook wykonuje fine-tuning wstępnie wytrenowanego modelu BERT (z notebooka 04) do klasyfikacji pacjentów na 5 kategorii triage.

### 🔄 Wymagania
- `medical_bert_mlm_*` - wstępnie wytrenowany model MLM (z notebooka 04)
- `corpus_combined_*.txt` - połączone dane z Synthea i MIMIC (z notebooka 03)

### 📊 Metodologia

#### 1. Format danych wejściowych
- Każda linia w `corpus_combined_*.txt` ma format: `[token1] [token2] ... [TARGET_LABEL]`
- Tokeny reprezentują: wiek, płeć, wyniki badań laboratoryjnych (kwantyzowane), objawy
- Etykiety: `TARGET_POZ`, `TARGET_SPEC_GASTROLOG`, `TARGET_SPEC_HEMATOLOG`, `TARGET_SPEC_NEFROLOG`, `TARGET_SPEC_SOR`

#### 2. Podział danych
- **Proporcje:** 80% train / 20% val
- **Metoda:** `train_test_split` 80/20 (bez stratify — multi-label)
- **Reprodukowalność:** `random_state=42` (deterministyczny podział)
- **Walidacja:** Sprawdzanie, czy wszystkie klasy są reprezentowane w train i val
- **Zapis:** Train/val split zapisywany do plików dla reprodukowalności

#### 3. Parametry modelu
- **Architektura:** `BertForSequenceClassification` z `problem_type="multi_label_classification"` (sigmoid)
- **Liczba etykiet:** NUM_LABELS (np. 8: POZ, GASTROLOG, HEMATOLOG, NEFROLOG, SOR, KARDIOLOG, PULMONOLOG, HEPATOLOG)
- **Max length:** 128 tokenów (wystarczające dla sekwencji medycznych)
- **Batch size:** 32 (train), 64 (eval) - optymalne dla pamięci GPU/CPU
- **Learning rate:** 2e-5 (standardowa wartość dla fine-tuning BERT)
- **Epochs:** 5 (wystarczające dla konwergencji)
- **Weight decay:** 0.01 (regularyzacja L2)

#### 4. Cost-Sensitive Learning z Clinical Cost Matrix
- **Uzasadnienie medtech:** False Negatives są krytyczne dla bezpieczeństwa pacjentów
  - HEMATOLOG → POZ: opóźniona diagnoza białaczki może być śmiertelna (koszt: 10.0)
  - SOR → POZ: pacjent wymagający pilnej opieki błędnie odesłany (koszt: 8.0)
  - POZ → SPEC: false positive, mniej kosztowne (koszt: 2-5)
- **Implementacja:** `MultiLabelBCETrainer` z **BCEWithLogitsLoss** + `pos_weight` (cost-sensitive per class)
- **Clinical Cost Matrix:** Definiuje koszty błędów z perspektywy bezpieczeństwa pacjentów
- **Dynamiczne wagi:** Kombinacja inverse frequency × cost-based weights
  - Inverse frequency: uwzględnia rzadkość klas
  - Cost-based: uwzględnia koszty kliniczne błędów
- **Formuła:** `CSFL = FL * cost_weight`, gdzie:
  - `FL = -alpha * (1 - p_t)^gamma * log(p_t)` (Focal Loss)
  - `cost_weight` zależy od średniego kosztu błędu dla prawdziwej klasy
- **Parametry:** `alpha=combined_weights` (dynamiczne), `gamma=2.0`, `cost_matrix=CLINICAL_COST_MATRIX`
- **MPS compatibility:** Wagi i loss obliczane na CPU (kompatybilność z Apple Silicon)

#### 5. Threshold Tuning per Class
- **Problem:** Argmax nie uwzględnia różnych kosztów błędów dla różnych klas
- **Rozwiązanie:** Różne progi decyzyjne dla różnych klas (niższy próg dla rzadkich klas)
- **Metoda:** Optymalizacja progów (0.1-0.9) na zbiorze walidacyjnym, maksymalizacja F1 per class
- **Implementacja:** Po treningu, automatyczna optymalizacja progów dla każdej klasy
- **Output:** `optimal_thresholds.json` - zapisane progi dla użycia w produkcji

#### 6. Walidacja danych
- Sprawdzanie, czy wszystkie klasy są w train i val
- Statystyki rozkładu klas przed treningiem
- Walidacja po podziale danych

### 📝 Output
- `bert_triage_model_YYYYMMDD_HHMMSS/` - wytrenowany model
- `corpus_train_YYYYMMDD_HHMMSS.txt` - zbiór treningowy (80%)
- `corpus_val_YYYYMMDD_HHMMSS.txt` - zbiór walidacyjny (20%, używany w notebooku 07)

### ⚖️ Uzasadnienie Clinical Cost Matrix

Clinical Cost Matrix definiuje koszty różnych błędów klasyfikacji z perspektywy bezpieczeństwa pacjentów i skutków klinicznych:

**Krytyczne False Negatives (wysokie koszty):**
- **HEMATOLOG → POZ (koszt: 10.0)**: Opóźniona diagnoza białaczki lub innych chorób hematologicznych może być śmiertelna. Pacjent z niskim HGB/PLT błędnie odesłany do POZ traci krytyczny czas na leczenie.
- **SOR → POZ (koszt: 8.0)**: Pacjent wymagający pilnej opieki specjalistycznej błędnie odesłany do POZ może doświadczyć poważnych powikłań.

**Wzajemne pomyłki między specjalizacjami:**
- **HEMATOLOG ↔ NEFROLOG (koszt: 7.0)**: Wzajemne pomyłki są kosztowne, ponieważ obie specjalizacje wymagają pilnej interwencji, ale leczenie jest różne. NEFRO→HEMATO = 7.0 (zwiększone z 5.0) ze względu na ryzyko opóźnienia w leczeniu białaczki.
- **GASTROLOG ↔ inne (koszt: 2-4)**: Mniej krytyczne, ale nadal wymagają korekty.

**False Positives (niższe koszty):**
- **POZ → SPEC (koszt: 2-5)**: Mniej kosztowne niż False Negatives, ale nadal wymagają korekty, aby uniknąć niepotrzebnego obciążenia systemu.

Matrix jest używana w **Cost-Sensitive Focal Loss**, gdzie koszty błędów są uwzględniane w funkcji straty, priorytetyzując redukcję False Negatives w klasach krytycznych (HEMATOLOG, SOR).

In [ ]:
import os
import sys
from pathlib import Path

# Baza ścieżek: katalog notebooks (działa z dowolnego cwd)
_cwd = Path(os.getcwd())
if (_cwd / "models" / "mlm").exists():
    NOTEBOOKS_DIR = _cwd
elif (_cwd / "notebooks" / "models" / "mlm").exists():
    NOTEBOOKS_DIR = _cwd / "notebooks"
else:
    for _p in [_cwd] + list(_cwd.parents):
        if (_p / "models" / "mlm").exists():
            NOTEBOOKS_DIR = _p
            break
    else:
        NOTEBOOKS_DIR = _cwd
NOTEBOOKS_DIR = NOTEBOOKS_DIR.resolve()
os.chdir(NOTEBOOKS_DIR)
print(f"📂 Baza ścieżek: {NOTEBOOKS_DIR}")

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import BertForSequenceClassification, PreTrainedTokenizerFast, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import numpy as np
from datetime import datetime
import json

# --- KONFIGURACJA ---
MLM_DIR = NOTEBOOKS_DIR / "models" / "mlm"
mlm_models = sorted([d for d in MLM_DIR.glob("medical_bert_mlm_*") if d.is_dir()], reverse=True)
if not mlm_models:
    raise FileNotFoundError("Nie znaleziono modelu medical_bert_mlm_* w models/mlm/. Uruchom najpierw notebook 04.")
PRETRAINED_DIR = str(mlm_models[0])
print(f"📂 Znaleziono model MLM: {PRETRAINED_DIR}")

In [ ]:


# Znajdź najnowszy corpus_combined
corpus_files = sorted(glob.glob("corpus_combined_*.txt"), reverse=True)
if not corpus_files:
    raise FileNotFoundError("Nie znaleziono pliku corpus_combined_*.txt. Uruchom najpierw notebook 03_Merge_Datasets.ipynb")
CORPUS_FILE = corpus_files[0]
print(f"📂 Używam corpus: {CORPUS_FILE}")

START_TIME = datetime.now()
OUTPUT_DIR = f"bert_triage_model_{START_TIME.strftime('%Y%m%d_%H%M%S')}"

# Mapowanie klas (wspólny moduł dla wszystkich notebooków)
import sys
from pathlib import Path

# Znajdź główny katalog projektu (gdzie jest label_mapping.py)
# Szukamy w bieżącym katalogu i katalogach nadrzędnych
current_dir = Path(os.getcwd())
project_root = None

# Sprawdź czy jesteśmy w głównym katalogu projektu
if (current_dir / "label_mapping.py").exists():
    project_root = current_dir
elif (current_dir / "notebooks" / "label_mapping.py").exists():
    # Jeśli jesteśmy w notebooks, idź w górę
    project_root = current_dir.parent
else:
    # Szukaj w katalogach nadrzędnych
    for parent in current_dir.parents:
        if (parent / "label_mapping.py").exists():
            project_root = parent
            break

if project_root is None:
    raise FileNotFoundError(
        f"Nie znaleziono pliku label_mapping.py. "
        f"Szukano w: {current_dir} i katalogach nadrzędnych."
    )

# Dodaj główny katalog projektu do sys.path
sys.path.insert(0, str(project_root))
print(f"📂 Dodano do sys.path: {project_root}")

from label_mapping import (
    LABEL_MAP,  # LABEL_TO_ID (kompatybilność wsteczna)
    ID_TO_LABEL,
    ID_TO_LABEL_SHORT,
    LABEL_NAMES,
    NUM_LABELS
)
ID2LABEL = ID_TO_LABEL  # Alias dla kompatybilności
# ⚠️ FIX dla MPS (Mac Apple Silicon) - MPS ma problemy z niektórymi operacjami
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    # MPS ma problemy z niektórymi operacjami loss, używamy CPU
    DEVICE = "cpu"
    print("⚠️ MPS dostępne, ale używamy CPU (kompatybilność z class weights)")
else:
    DEVICE = "cpu"

print(f"📥 Ładowanie modelu z: {PRETRAINED_DIR}")

# 1. Ładowanie Tokenizera (lokalny plik, nie z HuggingFace Hub)
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    PRETRAINED_DIR,
    local_files_only=True  # ⚠️ WAŻNE: Tylko lokalne pliki, nie próbuj pobierać z Hub
)

# 2. Parsowanie Danych (MULTI-LABEL: wszystkie etykiety z linii → wektor 0/1)
texts = []
labels = []  # lista wektorów numpy (NUM_LABELS,) z wartościami 0/1
label_keys = set(LABEL_MAP.keys())

print("📂 Przetwarzanie pliku danych (multi-label)...")
with open(CORPUS_FILE, "r") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        parts = line.split()
        line_labels = [p for p in parts if p in label_keys]
        line_text_parts = [p for p in parts if p not in label_keys]
        if not line_labels: continue
        input_text = " ".join(line_text_parts)
        label_vec = np.zeros(NUM_LABELS, dtype=np.float32)
        for l in line_labels:
            label_vec[LABEL_MAP[l]] = 1.0
        texts.append(input_text)
        labels.append(label_vec)

# Podział Train/Val (80/20; bez stratify — multi-label)
print("📊 Dzielenie danych na train/val (80/20)...")
# ============================================
# 🔧 DATA AUGMENTATION: Więcej przykładów dla przypadków granicznych
# ============================================
# Problem: Model myli NEFROLOG z HEMATOLOG (40% pomyłek)
# Rozwiązanie: Dodajemy więcej przykładów z kombinacjami tokenów

def augment_data_for_edge_cases(texts, labels):
    """Multi-label: labels to lista wektorów (N, NUM_LABELS). Duplikujemy wybrane przykłady."""
    NEFROLOG_ID = LABEL_MAP['TARGET_SPEC_NEFROLOG']
    HEMATOLOG_ID = LABEL_MAP['TARGET_SPEC_HEMATOLOG']
    augmented_texts = list(texts)
    augmented_labels = list(labels)
    augmentation_stats = {'NEFROLOG_augmented': 0, 'HEMATOLOG_augmented': 0}
    for text, label_vec in zip(texts, labels):
        tokens = text.split()
        has_high_creat = any('CREATININE_Q8' in t or 'CREATININE_Q9' in t or 'CREATININE_Q10' in t or 'CREATININE_HIGH' in t for t in tokens)
        has_high_urea = any('UREA_Q8' in t or 'UREA_Q9' in t or 'UREA_Q10' in t or 'UREA_HIGH' in t for t in tokens)
        has_low_hgb = any('HGB_Q1' in t or 'HGB_Q2' in t for t in tokens)
        has_low_plt = any('PLT_Q1' in t or 'PLT_Q2' in t for t in tokens)
        if label_vec[NEFROLOG_ID] == 1 and (has_high_creat or has_high_urea) and (has_low_hgb or has_low_plt):
            augmented_texts.append(text)
            augmented_labels.append(label_vec.copy())
            augmented_texts.append(text)
            augmented_labels.append(label_vec.copy())
            augmentation_stats['NEFROLOG_augmented'] += 2
        elif label_vec[HEMATOLOG_ID] == 1 and (has_low_hgb or has_low_plt) and not (has_high_creat or has_high_urea):
            augmented_texts.append(text)
            augmented_labels.append(label_vec.copy())
            augmentation_stats['HEMATOLOG_augmented'] += 1
    print(f"\n📊 Data Augmentation: NEFROLOG +{augmentation_stats['NEFROLOG_augmented']}, HEMATOLOG +{augmentation_stats['HEMATOLOG_augmented']}, łącznie {len(augmented_texts)} (było {len(texts)})")
    return augmented_texts, augmented_labels

print("\n🔧 Stosowanie data augmentation...")
texts_augmented, labels_augmented = augment_data_for_edge_cases(texts, labels)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts_augmented, labels_augmented, test_size=0.2, random_state=42
)
print(f"✅ Podział zakończony: {len(train_texts)} treningowych, {len(val_texts)} walidacyjnych.")

# Walidacja podziału danych
def validate_data_split(train_labels, val_labels, train_texts, val_texts):
    """Walidacja podziału dla multi-label (labels = lista wektorów 0/1)."""
    if len(train_texts) == 0 or len(val_texts) == 0:
        raise ValueError("❌ Train lub val jest pusty!")
    train_arr = np.array(train_labels)
    val_arr = np.array(val_labels)
    train_counts = train_arr.sum(axis=0)
    val_counts = val_arr.sum(axis=0)
    for c in range(NUM_LABELS):
        if train_counts[c] == 0: raise ValueError(f"❌ Brak klasy {ID_TO_LABEL[c]} w train")
        if val_counts[c] == 0: raise ValueError(f"❌ Brak klasy {ID_TO_LABEL[c]} w val")
    print("✅ Walidacja zakończona pomyślnie!")
    print(f"📊 Liczność klas w train: {dict(zip([ID_TO_LABEL_SHORT[i] for i in range(NUM_LABELS)], train_counts.astype(int)))}")
    print(f"📊 Liczność klas w val: {dict(zip([ID_TO_LABEL_SHORT[i] for i in range(NUM_LABELS)], val_counts.astype(int)))}")
    return train_counts, val_counts

# Wykonanie walidacji
train_dist, val_dist = validate_data_split(train_labels, val_labels, train_texts, val_texts)

# Zapis train/val split do plików (dla reprodukowalności i użycia w notebooku 07)
SPLIT_TIMESTAMP = START_TIME.strftime("%Y%m%d_%H%M%S")
train_split_file = f"corpus_train_{SPLIT_TIMESTAMP}.txt"
val_split_file = f"corpus_val_{SPLIT_TIMESTAMP}.txt"

print(f"\n💾 Zapisuję podział danych do plików (multi-label: wiele etykiet na linię)...")
with open(train_split_file, "w", encoding="utf-8") as f:
    for text, label_vec in zip(train_texts, train_labels):
        label_tokens = " ".join(ID2LABEL[i] for i in range(NUM_LABELS) if label_vec[i] == 1)
        f.write(f"{text} {label_tokens}\n")

with open(val_split_file, "w", encoding="utf-8") as f:
    for text, label_vec in zip(val_texts, val_labels):
        label_tokens = " ".join(ID2LABEL[i] for i in range(NUM_LABELS) if label_vec[i] == 1)
        f.write(f"{text} {label_tokens}\n")

print(f"✅ Zapisano train: {train_split_file} ({len(train_texts):,} próbek)")
print(f"✅ Zapisano val: {val_split_file} ({len(val_texts):,} próbek)")
print(f"📝 Notebook 07 użyje TYLKO pliku: {val_split_file}")

# Wagi klas dla multi-label: liczność pozytywów per klasa w train
train_labels_arr = np.array(train_labels)
train_label_counts = train_labels_arr.sum(axis=0)
total_samples = len(train_labels)
inverse_freq_weights = np.array([total_samples / (max(1, train_label_counts[c]) * NUM_LABELS) for c in range(NUM_LABELS)])

# --- COST-SENSITIVE LEARNING: CLINICAL COST MATRIX ---
# Z perspektywy medtech: False Negatives są krytyczne dla bezpieczeństwa pacjentów
# Cost matrix definiuje koszty różnych błędów z perspektywy klinicznej

# Definicja Clinical Cost Matrix:
# cost_matrix[i][j] = koszt sklasyfikowania klasy i jako klasa j
# Wysokie koszty dla False Negatives (np. HEMATOLOG → POZ = bardzo niebezpieczne)

# Macierz 8x8: POZ, GASTRO, HEMATO, NEFRO, SOR, KARDIO, PULMO, HEPATO
CLINICAL_COST_MATRIX = np.array([
    [0.0, 2.0, 5.0, 3.0, 4.0, 3.0, 3.0, 3.0],   # Prawda: POZ
    [3.0, 0.0, 4.0, 2.0, 3.0, 3.0, 3.0, 3.0],   # GASTROLOG
    [10.0, 6.0, 0.0, 7.0, 4.0, 4.0, 4.0, 5.0], # HEMATOLOG (FN→POZ krytyczne)
    [4.0, 3.0, 7.0, 0.0, 3.0, 4.0, 4.0, 4.0],  # NEFROLOG
    [8.0, 5.0, 4.0, 3.0, 0.0, 4.0, 4.0, 4.0],  # SOR (FN→POZ bardzo niebezpieczne)
    [4.0, 3.0, 4.0, 4.0, 4.0, 0.0, 3.0, 3.0],  # KARDIOLOG
    [4.0, 3.0, 4.0, 4.0, 4.0, 3.0, 0.0, 3.0],  # PULMONOLOG
    [4.0, 3.0, 5.0, 4.0, 4.0, 3.0, 3.0, 0.0],  # HEPATOLOG
])

# Uzasadnienie kosztów:
# - HEMATOLOG → POZ: 10.0 (najwyższy) - opóźniona diagnoza białaczki może być śmiertelna
# - SOR → POZ: 8.0 (wysoki) - pacjent wymagający pilnej opieki błędnie odesłany
# - HEMATOLOG ↔ NEFROLOG: 7.0 (zwiększone z 5.0) - ważne rozróżnienie kliniczne
#   * HEMATOLOG: niskie HGB/PLT → anemia, białaczka
#   * NEFROLOG: wysokie CREATININE/UREA → niewydolność nerek
#   * Pomyłka może prowadzić do niewłaściwego leczenia
# - POZ → SPEC: 2-5 (umiarkowany) - false positive, ale pacjent i tak trafi do specjalisty
# - POZ → POZ: 0.0 (brak błędu)

print("📊 CLINICAL COST MATRIX:")
print("   (wiersz = prawdziwa klasa, kolumna = predykcja)")
print("   Wysokie koszty dla False Negatives (krytyczne dla bezpieczeństwa)")
for i, class_name in enumerate([ID_TO_LABEL_SHORT[j] for j in range(NUM_LABELS)]):
    print(f"   {class_name:12s}: {CLINICAL_COST_MATRIX[i]}")

# Cost-based weights (średni koszt błędu dla każdej klasy)
# Dla każdej klasy: średni koszt, jeśli model się pomyli
cost_based_weights = np.array([
    CLINICAL_COST_MATRIX[i].mean() for i in range(NUM_LABELS)
])

# Normalizacja cost_based_weights (żeby były w podobnym zakresie)
cost_based_weights = cost_based_weights / cost_based_weights.min()

# Kombinacja: inverse frequency * cost-based (ważymy rzadkie klasy + klasy z wysokimi kosztami błędów)
combined_weights = inverse_freq_weights * cost_based_weights
combined_weights = combined_weights / combined_weights.min()  # Normalizacja do zakresu [1.0, ...]

print(f"\n⚖️ Inverse frequency weights: {inverse_freq_weights}")
print(f"⚖️ Cost-based weights: {cost_based_weights}")
print(f"⚖️ Combined weights (inverse_freq × cost): {combined_weights}")

# Używamy combined weights
manual_weights = combined_weights.tolist()
# ⚠️ FIX: class_weights zawsze na CPU (MPS ma problemy z CrossEntropyLoss + weights)
class_weights = torch.tensor(manual_weights, dtype=torch.float)
print(f"\n⚖️ Zastosowano COST-SENSITIVE wagi: {[f'{w:.2f}' for w in manual_weights]}")
print(f"⚖️ Wagi na urządzeniu: CPU (kompatybilność z loss function)")
# 3. Dataset z Sample Weights (ważenie przykładów z CREATININE/UREA)
class ClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128, compute_sample_weights=True):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels
        self.texts = texts  # Przechowujemy teksty dla obliczenia sample weights
        
        # Oblicz sample weights dla przykładów z ważnymi tokenami (CREATININE/UREA)
        if compute_sample_weights:
            self.sample_weights = self._compute_sample_weights(texts)
        else:
            self.sample_weights = torch.ones(len(labels), dtype=torch.float)
    
    def _compute_sample_weights(self, texts):
        """
        Oblicza wagi dla każdego przykładu na podstawie obecności ważnych tokenów.
        Przykłady z CREATININE/UREA mają wyższą wagę (ważniejsze dla rozróżnienia NEFROLOG).
        """
        weights = []
        for text in texts:
            tokens = text.split()
            weight = 1.0  # Domyślna waga
            
            # Zwiększ wagę jeśli są wysokie CREATININE/UREA (ważne dla NEFROLOG)
            has_high_creat = any('CREATININE_Q8' in t or 'CREATININE_Q9' in t or 'CREATININE_Q10' in t or 'CREATININE_HIGH' in t for t in tokens)
            has_high_urea = any('UREA_Q8' in t or 'UREA_Q9' in t or 'UREA_Q10' in t or 'UREA_HIGH' in t for t in tokens)
            
            if has_high_creat or has_high_urea:
                weight = 1.5  # 50% wyższa waga dla przykładów z CREATININE/UREA
            
            weights.append(weight)
        
        return torch.tensor(weights, dtype=torch.float)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)  # multi-label: wektor 0/1
        item['sample_weight'] = self.sample_weights[idx]
        return item

    def __len__(self):
        return len(self.labels)

# Dataset z sample weights (tylko dla train - zwiększamy wagę przykładów z CREATININE/UREA)
train_dataset = ClassificationDataset(train_texts, train_labels, tokenizer, compute_sample_weights=True)
val_dataset = ClassificationDataset(val_texts, val_labels, tokenizer, compute_sample_weights=False)  # Val bez sample weights

# 4. Model (multi-label: sigmoid na wyjściu)
model = BertForSequenceClassification.from_pretrained(
    PRETRAINED_DIR,
    num_labels=NUM_LABELS,
    id2label=ID_TO_LABEL,
    label2id=LABEL_MAP,
    problem_type="multi_label_classification",
    local_files_only=True
).to(DEVICE)

# 5. Metryki (multi-label: Hamming loss, F1 samples)
from sklearn.metrics import hamming_loss
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1.0 / (1.0 + np.exp(-np.array(logits)))
    preds = (probs >= 0.5).astype(np.float32)
    return {"hamming_loss": float(hamming_loss(labels, preds)), "f1_samples": float(f1_score(labels, preds, average='samples', zero_division=0))}

# --- MULTI-LABEL: BCE + pos_weight (cost-sensitive) ---
pos_weight = torch.tensor(combined_weights, dtype=torch.float)  # wyższa waga dla pozytywów w klasach SOR/HEMATOLOG

class MultiLabelBCETrainer(Trainer):
    """Trainer z BCEWithLogitsLoss i pos_weight (cost-sensitive dla multi-label)."""
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")  # [batch, NUM_LABELS]
        outputs = model(**inputs)
        logits = outputs.get("logits")
        logits_cpu = logits.cpu()
        labels_cpu = labels.cpu()
        loss = F.binary_cross_entropy_with_logits(logits_cpu, labels_cpu, pos_weight=pos_weight, reduction='mean')
        return (loss.to(logits.device), outputs) if return_outputs else loss.to(logits.device)

# 6. Trening
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch", # Poprawione nazewnictwo
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="none"
)

# Multi-label: BCE + pos_weight (cost-sensitive)
trainer = MultiLabelBCETrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("🚀 Start Fine-tuning (multi-label, BCE + pos_weight)...")
print("📊 Loss parametry:")
print(f"   - BCEWithLogitsLoss + pos_weight (cost-sensitive per class)")
print(f"   - Cost Matrix: uwzględnia koszty kliniczne błędów")
print(f"   - Priorytet: False Negatives w HEMATOLOG/SOR (bezpieczeństwo pacjentów)")
trainer.train()

# 7. Ewaluacja (multi-label: sigmoid, baseline = próg 0.5)
print("\n📊 Generowanie raportu (multi-label, baseline próg 0.5)...")
preds_output = trainer.predict(val_dataset)
y_probs = (1.0 / (1.0 + np.exp(-np.array(preds_output.predictions)))).astype(np.float32)
y_preds_baseline = (y_probs >= 0.5).astype(np.float32)
val_labels_arr = np.array(val_labels)

print("=== WYNIKI BASELINE (próg 0.5 per class) ===")
print(classification_report(val_labels_arr, y_preds_baseline, target_names=[ID_TO_LABEL[i] for i in range(NUM_LABELS)], zero_division=0))

# 8. Threshold Tuning per Class (optymalizacja progów dla rzadkich klas)
print("\n🎯 Threshold Tuning per Class...")
print("   Optymalizacja progów decyzyjnych dla każdej klasy (niższy próg dla rzadkich klas)")

from sklearn.metrics import f1_score

# Funkcja do optymalizacji progów
def optimize_thresholds(y_true, y_probs, class_names):
    """
    Optymalizuje progi decyzyjne dla każdej klasy osobno.
    Dla rzadkich klas (np. HEMATOLOG) używa niższego progu.
    """
    thresholds = {}
    best_f1_scores = {}
    
    # Dla każdej klasy znajdź optymalny próg
    for class_id in range(NUM_LABELS):
        class_name = ID_TO_LABEL_SHORT[class_id]
        best_threshold = 0.5
        best_f1 = 0
        
        # Przeszukaj progi od 0.1 do 0.9
        for threshold in np.arange(0.1, 0.95, 0.05):
            # Dla tej klasy: jeśli prawdopodobieństwo > threshold, przewiduj tę klasę
            y_pred_class = (y_probs[:, class_id] > threshold).astype(int)
            y_true_class = y_true[:, class_id].astype(int)
            f1 = f1_score(y_true_class, y_pred_class, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold
        
        thresholds[class_id] = best_threshold
        best_f1_scores[class_id] = best_f1
        print(f"   {class_name}: próg={best_threshold:.2f}, F1={best_f1:.3f}")
    
    return thresholds, best_f1_scores

# Optymalizacja progów
optimal_thresholds, threshold_f1s = optimize_thresholds(
    val_labels_arr,
    y_probs, 
    [ID_TO_LABEL_SHORT[i] for i in range(NUM_LABELS)]
)

# Predykcje z optymalizowanymi progami
def predict_with_thresholds(y_probs, thresholds):
    """Multi-label: dla każdej klasy osobno stosuje próg. Zwraca (n, NUM_LABELS) 0/1."""
    n = len(y_probs)
    y_pred = np.zeros((n, NUM_LABELS), dtype=np.float32)
    for class_id in range(NUM_LABELS):
        y_pred[:, class_id] = (y_probs[:, class_id] > thresholds[class_id]).astype(np.float32)
    return y_pred

y_preds_threshold = predict_with_thresholds(y_probs, optimal_thresholds)

print("\n=== WYNIKI Z THRESHOLD TUNING ===")
print(classification_report(val_labels_arr, y_preds_threshold, target_names=[ID_TO_LABEL[i] for i in range(NUM_LABELS)], zero_division=0))

# Porównanie F1 scores (multi-label: per-class)
print("\n📊 PORÓWNANIE F1 SCORES (baseline vs threshold):")
for class_id in range(NUM_LABELS):
    class_name = ID_TO_LABEL_SHORT[class_id]
    y_true_binary = val_labels_arr[:, class_id].astype(int)
    y_pred_baseline_binary = y_preds_baseline[:, class_id].astype(int)
    y_pred_threshold_binary = y_preds_threshold[:, class_id].astype(int)
    
    f1_baseline = f1_score(y_true_binary, y_pred_baseline_binary, zero_division=0)
    f1_threshold = f1_score(y_true_binary, y_pred_threshold_binary, zero_division=0)
    improvement = f1_threshold - f1_baseline
    print(f"   {class_name}: baseline={f1_baseline:.3f}, threshold={f1_threshold:.3f} ({improvement:+.3f})")

# Zapisujemy optymalne progi do pliku
import json
thresholds_file = Path(OUTPUT_DIR) / "optimal_thresholds.json"
with open(thresholds_file, "w") as f:
    json.dump({ID_TO_LABEL_SHORT[k]: float(v) for k, v in optimal_thresholds.items()}, f, indent=2)
print(f"\n💾 Zapisano optymalne progi do: {thresholds_file}")

# Zapisujemy cost matrix i wagi do pliku (dla dokumentacji i audytu)
cost_matrix_file = Path(OUTPUT_DIR) / "clinical_cost_matrix.json"
cost_matrix_dict = {
    "cost_matrix": CLINICAL_COST_MATRIX.tolist(),
    "class_names": [ID_TO_LABEL_SHORT[i] for i in range(NUM_LABELS)],
    "justification": {
        "HEMATOLOG_POZ": "10.0 - Opóźniona diagnoza białaczki może być śmiertelna",
        "SOR_POZ": "8.0 - Pacjent wymagający pilnej opieki błędnie odesłany",
        "POZ_SPEC": "2-5 - False positive, mniej kosztowne (pacjent i tak trafi do specjalisty)"
    },
    "weights": {
        "inverse_frequency": inverse_freq_weights.tolist(),
        "cost_based": cost_based_weights.tolist(),
        "combined": combined_weights.tolist()
    }
}
with open(cost_matrix_file, "w") as f:
    json.dump(cost_matrix_dict, f, indent=2)
print(f"💾 Zapisano cost matrix do: {cost_matrix_file}")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"🏆 Finalny model zapisany w: {OUTPUT_DIR}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# Multi-label: predictions to (n, NUM_LABELS) 0/1
if 'y_preds_threshold' in globals():
    predictions = y_preds_threshold
    title_suffix = " (z Threshold Tuning)"
elif 'y_preds_baseline' in globals():
    predictions = y_preds_baseline
    title_suffix = " (baseline 0.5)"
else:
    raise NameError("Uruchom najpierw Cell 1 (trening modelu).")

# Macierz pomyłek multi-label: cm[i,j] = liczba próbek gdzie true ma klasę i i pred ma klasę j
val_labels_arr_cm = np.array(val_labels)
cm = np.zeros((NUM_LABELS, NUM_LABELS))
for i in range(NUM_LABELS):
    for j in range(NUM_LABELS):
        cm[i, j] = np.sum((val_labels_arr_cm[:, i] == 1) & (predictions[:, j] == 1))
row_sums = cm.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
cm_norm = cm.astype('float') / row_sums

class_names = [ID_TO_LABEL_SHORT[i] for i in range(NUM_LABELS)]

plt.figure(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('Prawdziwa Klasa (Lekarz)')
plt.xlabel('Predykcja Modelu (AI)')
plt.title(f'Macierz pomyłek{title_suffix}')
plt.tight_layout()
plt.show()

# Wyświetl również bezwzględne liczby
print("\n📊 Macierz pomyłek (bezwzględne liczby):")
print(cm)

In [ ]:
# ============================================
# 🔧 POST-PROCESSING RULES: Korekta HEMATOLOG ↔ NEFROLOG
# ============================================
# Problem: Model myli HEMATOLOG z NEFROLOG (i odwrotnie)
# Rozwiązanie: Deterministyczne reguły oparte na tokenach laboratoryjnych
# 
# Logika:
# - HEMATOLOG: niskie HGB (HGB_Q1, HGB_Q2) lub niskie PLT (PLT_Q1, PLT_Q2)
# - NEFROLOG: wysokie CREATININE (CREATININE_HIGH) lub wysokie UREA (UREA_HIGH)
# 
# Multi-label: y_preds to (n, NUM_LABELS) 0/1; korygujemy bity HEMATOLOG/NEFROLOG

def apply_post_processing_rules(texts, y_preds, y_probs):
    """Korekta HEMATOLOG ↔ NEFROLOG. y_preds: (n, NUM_LABELS) 0/1."""
    HEMATOLOG_ID = LABEL_MAP['TARGET_SPEC_HEMATOLOG']
    NEFROLOG_ID = LABEL_MAP['TARGET_SPEC_NEFROLOG']
    y_preds_corrected = np.array(y_preds, dtype=np.float32, copy=True)
    corrections = {'HEMATOLOG→NEFROLOG': 0, 'NEFROLOG→HEMATOLOG': 0}
    
    for i, (text, pred, probs) in enumerate(zip(texts, y_preds, y_probs)):
        tokens = text.split()
        has_low_hgb = any('HGB_Q1' in t or 'HGB_Q2' in t for t in tokens)
        has_low_plt = any('PLT_Q1' in t or 'PLT_Q2' in t for t in tokens)
        has_low_wbc = any('WBC_Q1' in t or 'WBC_Q2' in t for t in tokens)
        hematolog_indicators = (has_low_hgb or has_low_plt or has_low_wbc)
        has_high_creatinine = any('CREATININE_Q8' in t or 'CREATININE_Q9' in t or 'CREATININE_Q10' in t for t in tokens)
        has_high_urea = any('UREA_Q8' in t or 'UREA_Q9' in t or 'UREA_Q10' in t for t in tokens)
        has_high_creatinine_alt = any('CREATININE_HIGH' in t for t in tokens)
        has_high_urea_alt = any('UREA_HIGH' in t for t in tokens)
        nefrolog_indicators = has_high_creatinine or has_high_urea or has_high_creatinine_alt or has_high_urea_alt
        
        if pred[NEFROLOG_ID] == 1 and hematolog_indicators and not nefrolog_indicators and probs[HEMATOLOG_ID] > 0.2:
            y_preds_corrected[i, NEFROLOG_ID] = 0
            y_preds_corrected[i, HEMATOLOG_ID] = 1
            corrections['NEFROLOG→HEMATOLOG'] += 1
        elif pred[HEMATOLOG_ID] == 1 and nefrolog_indicators and probs[NEFROLOG_ID] > 0.2:
            y_preds_corrected[i, HEMATOLOG_ID] = 0
            y_preds_corrected[i, NEFROLOG_ID] = 1
            corrections['HEMATOLOG→NEFROLOG'] += 1
    return y_preds_corrected, corrections

# Zastosuj post-processing do predykcji z threshold tuning
print("🔧 Stosowanie post-processing rules (korekta HEMATOLOG ↔ NEFROLOG)...")
y_preds_corrected, corrections = apply_post_processing_rules(
    val_texts, y_preds_threshold, y_probs
)

print(f"\n📊 Korekty wykonane:")
print(f"   NEFROLOG → HEMATOLOG: {corrections['NEFROLOG→HEMATOLOG']} przypadków")
print(f"   HEMATOLOG → NEFROLOG: {corrections['HEMATOLOG→NEFROLOG']} przypadków")
print(f"   Łącznie: {sum(corrections.values())} korekt")

# Porównanie wyników
print("\n=== WYNIKI Z POST-PROCESSING ===\n")
print(classification_report(
    val_labels_arr, y_preds_corrected, 
    target_names=[ID_TO_LABEL[i] for i in range(NUM_LABELS)], zero_division=0
))

# Porównanie F1 scores przed i po korekcie
print("\n📊 PORÓWNANIE F1 SCORES (przed vs po post-processing):")
for class_id in range(NUM_LABELS):
    class_name = ID_TO_LABEL_SHORT[class_id]
    y_true_binary = val_labels_arr[:, class_id].astype(int)
    y_pred_before_binary = y_preds_threshold[:, class_id].astype(int)
    y_pred_after_binary = y_preds_corrected[:, class_id].astype(int)
    
    f1_before = f1_score(y_true_binary, y_pred_before_binary, zero_division=0)
    f1_after = f1_score(y_true_binary, y_pred_after_binary, zero_division=0)
    improvement = f1_after - f1_before
    
    print(f"   {class_name}: przed={f1_before:.3f}, po={f1_after:.3f} ({improvement:+.3f})")

# Zaktualizuj y_preds_threshold na skorygowaną wersję (dla dalszych analiz)
y_preds_threshold = y_preds_corrected
print("\n✅ Post-processing zakończony. y_preds_threshold został zaktualizowany.")